In [ ]:
# notebook adapted from https://github.com/chung-neuroai-lab/SNAP/

import brainscore_vision.benchmarks.majajhong2015.benchmark as majajhong2015
from tqdm import tqdm 
import pandas as pd
import numpy as np
import os

def get_brainscore(identifier="majajhong2015", region='IT'):

    if identifier == "majajhong2015":
        import brainscore_vision.benchmarks.majajhong2015.benchmark as majajhong2015
        assert region in ['IT', 'V4']

        neural_data = majajhong2015.load_assembly(average_repetitions=False, region=region, access='public')
        visual_degree = 8
    else:
        raise Exception(f"Only {['freemanziemba2013']} allowed.")

    # Get individual image ids
    image_ids = list(set(neural_data.image_id.data))

    # Create dictionary for repetitions, responses, image files, names and categories for each distinct image
    repetitions = {}
    neural_responses = {}
    image_files = {}
    image_names = {}
    image_categories = {}
    image_backgrounds = {}
    animals = {}
    subregions = {}


    from brainscore_vision.benchmark_helpers.screen import place_on_screen
    stimulus_set = place_on_screen(neural_data.stimulus_set,
                                   target_visual_degrees=8,
                                   source_visual_degrees=visual_degree)

    for image_id in tqdm(image_ids):

        data_image = neural_data.sel(image_id=image_id)

        repetition = len(data_image.presentation.repetition)
        neural_response = data_image.values.squeeze()
        image_file = stimulus_set.stimulus_paths[image_id]

        if identifier == "majajhong2015":
            image_name = data_image.object_name.values
            image_category = data_image.category_name.values
            image_background = data_image.background_id.values
            animal = data_image.animal.values
            subregion = data_image.subregion.values

        elif identifier == "freemanziemba2013":
            image_name = data_image.texture_family.values
            image_category = data_image.texture_type.values
        else:
            raise Exception(f"Only {['majajhong2015', 'freemanziemba2013']} allowed.")

        repetitions[image_id] = repetition
        neural_responses[image_id] = neural_response
        image_files[image_id] = image_file
        image_names[image_id] = image_name[0]
        image_categories[image_id] = image_category[0]
        image_backgrounds[image_id] = image_background[0]
        animals[image_id] = animal
        subregions[image_id] = subregion

    # Create pandas dataframe
    data = {'image_ids': image_ids,
            'repetition': repetitions.values(),
            'neural_responses': neural_responses.values(),
            'image_names': image_names.values(),
            'image_categories': image_categories.values(),
            'image_files': image_files.values(),
            "image_backgrounds": image_backgrounds.values(),
            "animals": animals.values(),
            "subregions": subregions.values(),
           }

    # Sort according to categories
    df = pd.DataFrame(data=data)
    df = df.sort_values(by=['image_categories', 'image_files'], ascending=[True, True], ignore_index=True)

    # Compute mean_responses and extract features
    df.insert(2, "mean_responses", df.neural_responses.map(lambda x: x.mean(0)))
    df.insert(3, "std_responses", df.neural_responses.map(lambda x: x.std(0)))

    return df


In [ ]:
data_root = f"/path/to/data_root"
df_IT = get_brainscore(region = "IT")
df_IT.to_pickle(f'{data_root}/df_majajhong2015_IT.pkl')

df_V4 = get_brainscore(region = "V4")
df_V4.to_pickle(f'{data_root}/df_majajhong2015_V4.pkl')


In [ ]:
from brainscore_vision.model_helpers.activations.pca import _get_imagenet_val

file_paths = _get_imagenet_val(1000)
np.savez(f"{data_root}/Imagenet_Val_1000_filepaths.npz", 
         file_paths=file_paths)